# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [1]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة المفاتيح السرية من Kaggle Secrets
user_secrets = UserSecretsClient()
openrouter_key = user_secrets.get_secret("OPENROUTER_API_KEY")
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")

# 2. تهيئة عميل OpenRouter (للنماذج الرخيصة)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

# 3. تهيئة عميل Inception Labs (لنموذج Mercury)
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key,
)

print("✅ تم الاتصال بالمنصتين بنجاح!")
print("📡 OpenRouter: جاهز للنماذج الرخيصة")
print("📡 Inception Labs: جاهز لـ Mercury")

✅ تم الاتصال بالمنصتين بنجاح!
📡 OpenRouter: جاهز للنماذج الرخيصة
📡 Inception Labs: جاهز لـ Mercury


In [4]:
!pip install --upgrade torchaudio -q
print("✅ تم تحديث torchaudio ليتوافق مع PyTorch!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.4 MB/s eta 0:00:00
✅ تم تحديث torchaudio ليتوافق مع PyTorch!


In [8]:


try:
    response = inception_client.chat.completions.create(
        model="mercury-2.5",
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        max_tokens=500,
        reasoning_effort="low"
    )
    
    content = response.choices[0].message.content
    
    if content:
        print(f"✅ نجح Mercury! الرد: '{content.strip()}'")
        print(f"\n📊 إحصائيات الرد:")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - prompt_tokens: {response.usage.prompt_tokens}")
            print(f"   - completion_tokens: {response.usage.completion_tokens}")
            print(f"   - total_tokens: {response.usage.total_tokens}")
    else:
        print("⚠️ المحتوى لا يزال فارغاً")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - total_tokens المستهلكة: {response.usage.total_tokens}")
    
except Exception as e:
    print(f"❌ فشل! السبب: {str(e)[:200]}")

✅ نجح Mercury! الرد: 'Hello! I hope you are having a wonderful day.'

📊 إحصائيات الرد:
   - finish_reason: stop
   - prompt_tokens: 6
   - completion_tokens: 280
   - total_tokens: 286


In [9]:
import requests, json, re, time, gc
import pandas as pd

# تحميل البريفات (نفس ما فعلنا في الكود السابق)
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"📋 عدد البريفات: {len(briefs)}\n")

def brief_to_prompt(b):
    return (
        f"Generate a brand name and a short tagline for this business.\n"
        f"Industry: {b['industry']}\n"
        f"Target audience: {b['target_audience']}\n"
        f"Brand purpose: {b['brand_purpose']}\n"
        f"Personality: {', '.join(b['personality'])}\n"
        f"Tone: {b['tone']}\n"
        f"Reply ONLY in this exact format: Name: ... | Tagline: ..."
    )

def clean_output(text):
    # إزالة أي تفكير داخلي أو تنسيق JSON
    if not text:
        return ""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    return text.strip()

# إعداد عميل Inception Labs
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key
)

mercury_results = []
failed_briefs = []

print(f"⏳ بدء اختبار Mercury 2.5 على {len(briefs)} بريف...\n")

for i, brief in enumerate(briefs):
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a branding expert. Generate a brand name and tagline."
            },
            {
                "role": "user",
                "content": brief_to_prompt(brief)
            }
        ]

        # استدعاء Mercury عبر Inception API
        response = inception_client.chat.completions.create(
            model="mercury-2.5",
            messages=messages,
            max_tokens=500,  # Mercury يحتاج توكنات أكثر بسبب Diffusion-based
            reasoning_effort="low"  # لتقليل التفكير الداخلي
        )

        # استخراج النتيجة
        raw_output = response.choices[0].message.content
        
        if raw_output:
            clean = clean_output(raw_output)
            mercury_results.append({
                "brief_id": brief["id"],
                "industry": brief["industry"],
                "model": "Mercury-2.5",
                "output": clean
            })
            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ✅")
        else:
            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ⚠️ محتوى فارغ")
            failed_briefs.append(brief["id"])

    except Exception as e:
        print(f"   [{i+1}/{len(briefs)}] {brief['id']} ❌ خطأ: {str(e)[:80]}")
        failed_briefs.append(brief["id"])

    # تأخير بسيط لتجنب Rate Limiting
    time.sleep(0.5)

print(f"\n✅ نجح: {len(mercury_results)}/{len(briefs)}")
print(f"❌ فشل: {len(failed_briefs)}/{len(briefs)}")

# حفظ النتائج
df_mercury = pd.DataFrame(mercury_results)
df_mercury.to_csv("mercury_results.csv", index=False)
print("\n💾 النتائج محفوظة بـ mercury_results.csv")
df_mercury.head(10)

📋 عدد البريفات: 30

⏳ بدء اختبار Mercury 2.5 على 30 بريف...

   [1/30] BR001 ✅
   [2/30] BR002 ✅
   [3/30] BR003 ✅
   [4/30] BR004 ✅
   [5/30] BR005 ✅
   [6/30] BR006 ✅
   [7/30] BR007 ✅
   [8/30] BR008 ✅
   [9/30] BR009 ✅
   [10/30] BR010 ✅
   [11/30] BR011 ✅
   [12/30] BR012 ✅
   [13/30] BR013 ✅
   [14/30] BR014 ✅
   [15/30] BR015 ✅
   [16/30] BR016 ✅
   [17/30] BR017 ✅
   [18/30] BR018 ✅
   [19/30] BR019 ✅
   [20/30] BR020 ✅
   [21/30] BR021 ✅
   [22/30] BR022 ✅
   [23/30] BR023 ✅
   [24/30] BR024 ✅
   [25/30] BR025 ✅
   [26/30] BR026 ✅
   [27/30] BR027 ✅
   [28/30] BR028 ✅
   [29/30] BR029 ✅
   [30/30] BR030 ✅

✅ نجح: 30/30
❌ فشل: 0/30

💾 النتائج محفوظة بـ mercury_results.csv


,brief_id,industry,model,output
0,BR001,coffee,Mercury-2.5,Name: The Grind Spot | Tagline: Fuel your stud...
1,BR002,healthy snacks,Mercury-2.5,Name: BrightBites | Tagline: Fresh fuel for yo...
2,BR003,specialty bakery,Mercury-2.5,Name: Hearth & Flour | Tagline: Traditional re...
3,BR004,specialty tea,Mercury-2.5,Name: Saffron & Sage | Tagline: Premium blends...
4,BR005,street food,Mercury-2.5,Name: BiteBlast | Tagline: Fuel your hustle fo...
5,BR006,sustainable fashion,Mercury-2.5,Name: ReVibe | Tagline: Wear the change
6,BR007,streetwear,Mercury-2.5,Name: UNBOUND | Tagline: Wear Your Chaos.
7,BR008,skincare,Mercury-2.5,"Name: Lumina | Tagline: Gentle skincare, simpl..."
8,BR009,perfume,Mercury-2.5,Name: Velatura | Tagline: Scent of soul and place
9,BR010,athleisure fashion,Mercury-2.5,Name: Kinetic | Tagline: Move Through Every Da...


In [5]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import traceback
import re
import gc

print(f"GPU: {torch.cuda.get_device_name(0)}")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen/Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LiquidAI/LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

results = {}  # عشان نخزن الردود ونقارنها بعدين

print("🔍 بدء اختبار النماذج محلياً على GPU...\n")

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    model = None  # نعرّفه هون عشان الـ finally يلاقيه حتى لو فشل التحميل بنفسه
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        messages = [{"role": "user", "content": "Generate a brand name and a short tagline for a modern coffee shop targeting university students. Reply in this format: Name: ... | Tagline: ..."}]

        try:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True,
                enable_thinking=False
            ).to(model.device)
        except TypeError:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True
            ).to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=300)
        input_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        clean_content = re.sub(r'.*</think>\s*', '', content, flags=re.DOTALL).strip()
        results[model_info["name"]] = clean_content
        print(f"   ✅ الرد: '{clean_content}'\n")

    except Exception as e:
        print(f"   ❌ فشل! نوع الخطأ: {type(e).__name__}")
        traceback.print_exc()
        results[model_info["name"]] = None
        print()

    finally:
        # هاد الجزء بينفّذ دايماً — نجح الموديل ولا فشل
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى الاختبار!\n")
print("📋 ملخص النتائج:")
for name, result in results.items():
    print(f"\n{name}:\n{result}")

GPU: Tesla T4
🔍 بدء اختبار النماذج محلياً على GPU...

⏳ جاري تحميل: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   ✅ الرد: 'Name: Brew Haven | Tagline: Your Daily Dose of Caffeine and Community'

⏳ جاري تحميل: LiquidAI/LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


   ✅ الرد: 'The user wants me to generate a brand name and a short tagline for a modern coffee shop targeting university students. The response must be in the specific format: "Name: ... | Tagline: ..."

I need to create:
1. A catchy, modern brand name suitable for a coffee shop aimed at university students (think trendy, energetic, student-friendly)
2. A concise tagline that captures the essence of the brand

Brainstorming names:
- "Campus Brew" - classic but maybe too generic
- "Study & Sip" - directly targets students
- "The Lecture Latte" - plays on academic lectures
- "Cup of Knowledge" - educational vibe
- "Grind & Grind" - puns on grinding beans and studying hard
- "Student Bean" - straightforward
- "The Study Spot" - functional
- "Perk & Paper" - combines coffee (perk) with academic papers
- "Caffeine Campus" - direct reference to campus life
- "The Morning Shift" - appeals to early morning study sessions
- "Brewed Scholars" - academic twist
- "Latte Lab" - scientific/modern fe

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, gc, re, json, requests
import pandas as pd

print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# تحميل البريفات مباشرة من GitHub
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"📋 عدد البريفات: {len(briefs)}\n")

def brief_to_prompt(b):
    return (
        f"Generate a brand name and a short tagline for this business.\n"
        f"Industry: {b['industry']}\n"
        f"Target audience: {b['target_audience']}\n"
        f"Brand purpose: {b['brand_purpose']}\n"
        f"Personality: {', '.join(b['personality'])}\n"
        f"Tone: {b['tone']}\n"
        f"Reply ONLY in this exact format: Name: ... | Tagline: ..."
    )

def clean_output(text):
    return re.sub(r'.*</think>\s*', '', text, flags=re.DOTALL).strip()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

all_results = []

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        for i, brief in enumerate(briefs):
            messages = [{"role": "user", "content": brief_to_prompt(brief)}]

            try:
                inputs = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True,
                    return_tensors="pt", return_dict=True,
                    enable_thinking=False
                ).to(model.device)
            except TypeError:
                inputs = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True,
                    return_tensors="pt", return_dict=True
                ).to(model.device)

            outputs = model.generate(**inputs, max_new_tokens=150)
            input_length = inputs["input_ids"].shape[-1]
            raw = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
            clean = clean_output(raw)

            all_results.append({
                "brief_id": brief["id"],
                "industry": brief["industry"],
                "model": model_info["name"],
                "output": clean
            })

            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ✅")

    except Exception as e:
        print(f"   ❌ فشل الموديل كله! {type(e).__name__}: {e}")

    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

    print()

# حفظ النتائج بجدول عشان تقارنيهم بسهولة
df = pd.DataFrame(all_results)
df.to_csv("comparison_results.csv", index=False)
print("✅ خلصت! النتائج محفوظة بـ comparison_results.csv")
df.head(10)

GPU: Tesla T4

📋 عدد البريفات: 30

⏳ جاري تحميل: Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   [1/30] BR001 ✅
   [2/30] BR002 ✅
   [3/30] BR003 ✅
   [4/30] BR004 ✅
   [5/30] BR005 ✅
   [6/30] BR006 ✅
   [7/30] BR007 ✅
   [8/30] BR008 ✅
   [9/30] BR009 ✅
   [10/30] BR010 ✅
   [11/30] BR011 ✅
   [12/30] BR012 ✅
   [13/30] BR013 ✅
   [14/30] BR014 ✅
   [15/30] BR015 ✅
   [16/30] BR016 ✅
   [17/30] BR017 ✅
   [18/30] BR018 ✅
   [19/30] BR019 ✅
   [20/30] BR020 ✅
   [21/30] BR021 ✅
   [22/30] BR022 ✅
   [23/30] BR023 ✅
   [24/30] BR024 ✅
   [25/30] BR025 ✅
   [26/30] BR026 ✅
   [27/30] BR027 ✅
   [28/30] BR028 ✅
   [29/30] BR029 ✅
   [30/30] BR030 ✅

⏳ جاري تحميل: LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

   [1/30] BR001 ✅
   [2/30] BR002 ✅
   [3/30] BR003 ✅
   [4/30] BR004 ✅
   [5/30] BR005 ✅
   [6/30] BR006 ✅
   [7/30] BR007 ✅
   [8/30] BR008 ✅
   [9/30] BR009 ✅
   [10/30] BR010 ✅
   [11/30] BR011 ✅
   [12/30] BR012 ✅
   [13/30] BR013 ✅
   [14/30] BR014 ✅
   [15/30] BR015 ✅
   [16/30] BR016 ✅
   [17/30] BR017 ✅
   [18/30] BR018 ✅
   [19/30] BR019 ✅
   [20/30] BR020 ✅
   [21/30] BR021 ✅
   [22/30] BR022 ✅
   [23/30] BR023 ✅
   [24/30] BR024 ✅
   [25/30] BR025 ✅
   [26/30] BR026 ✅
   [27/30] BR027 ✅
   [28/30] BR028 ✅
   [29/30] BR029 ✅
   [30/30] BR030 ✅

✅ خلصت! النتائج محفوظة بـ comparison_results.csv


,brief_id,industry,model,output
0,BR001,coffee,Qwen3-8B,"Name: Brew & Study | Tagline: Fuel your brain,..."
1,BR002,healthy snacks,Qwen3-8B,"Name: Snackify | Tagline: Fuel Your Day, Keep ..."
2,BR003,specialty bakery,Qwen3-8B,Name: Crumb & Co. | Tagline: Baked with Tradit...
3,BR004,specialty tea,Qwen3-8B,"Name: TerraVerve | Tagline: Sip the World, One..."
4,BR005,street food,Qwen3-8B,"Name: Flippy Fries | Tagline: Crave More, Pay ..."
5,BR006,sustainable fashion,Qwen3-8B,Name: Eclat Earth | Tagline: Style that grows ...
6,BR007,streetwear,Qwen3-8B,"Name: VerveX | Tagline: Wear Your Rebellion, O..."
7,BR008,skincare,Qwen3-8B,"Name: SereneSkin | Tagline: Gentle care, simpl..."
8,BR009,perfume,Qwen3-8B,Name: Eclat Éphémère | Tagline: Capture the Mo...
9,BR010,athleisure fashion,Qwen3-8B,"Name: VibeFit | Tagline: Move With Purpose, We..."


In [12]:
# دمج النتائج
df_qwen = pd.read_csv("comparison_results.csv")
df_mercury = pd.read_csv("mercury_results.csv")

# تأكد أن comparison_results.csv يحتوي فقط على Qwen و Liquid
df_all = pd.concat([df_qwen, df_mercury], ignore_index=True)
df_all.to_csv("all_models_comparison.csv", index=False)

print(f"✅ تم دمج {len(df_all)} نتيجة في ملف واحد!")
print(df_all.head(90))

✅ تم دمج 90 نتيجة في ملف واحد!
   brief_id                  industry        model  \
0     BR001                    coffee     Qwen3-8B   
1     BR002            healthy snacks     Qwen3-8B   
2     BR003          specialty bakery     Qwen3-8B   
3     BR004             specialty tea     Qwen3-8B   
4     BR005               street food     Qwen3-8B   
..      ...                       ...          ...   
85    BR026           travel planning  Mercury-2.5   
86    BR027                  pet care  Mercury-2.5   
87    BR028           interior design  Mercury-2.5   
88    BR029       digital photography  Mercury-2.5   
89    BR030  local community services  Mercury-2.5   

                                               output  
0   Name: Brew & Study | Tagline: Fuel your brain,...  
1   Name: Snackify | Tagline: Fuel Your Day, Keep ...  
2   Name: Crumb & Co. | Tagline: Baked with Tradit...  
3   Name: TerraVerve | Tagline: Sip the World, One...  
4   Name: Flippy Fries | Tagline: Crave 

In [13]:
import os
import json
import re
import time
import gc
import torch
import requests
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("🚀 بدء المقارنة العادلة بين النماذج الثلاثة\n")

# ═══════════════════════════════════════════════════════════
# 1. الإعدادات الأساسية (موحدة للجميع)
# ═══════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are an expert Brand Strategist.
Given a brand brief, generate a complete Brand Specification.
You MUST reply ONLY in valid JSON format, no markdown, no extra text."""

USER_PROMPT_TEMPLATE = """Analyze this brief and generate the specification:

Industry: {industry}
Target Audience: {target_audience}
Brand Purpose: {brand_purpose}
Personality: {personality}
Tone: {tone}

Generate a JSON object with EXACTLY this structure:
{{
  "brand_name": "string (creative name for the brand)",
  "tagline": "string (short memorable phrase)",
  "color_palette": ["color1", "color2", "color3"],
  "typography": "string (font style recommendation)",
  "visual_style": "string (design direction)",
  "personality_traits": ["trait1", "trait2", "trait3"]
}}"""

MAX_TOKENS = 500
TEMPERATURE = 0.3
NUM_BRIEFS = 5

# إعدادات quantization للنماذج المحلية
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

# ═══════════════════════════════════════════════════════════
# 2. تحميل البيانات واختيار 5 بريفات متنوعة
# ═══════════════════════════════════════════════════════════

print("📥 تحميل البيانات من GitHub...")
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
all_briefs = requests.get(url).json()

# اختيار أول 5 بريفات (نفسها لكل النماذج لضمان العدالة)
selected_briefs = all_briefs[:NUM_BRIEFS]
print(f"✅ تم اختيار {NUM_BRIEFS} بريفات للمقارنة:")
for i, b in enumerate(selected_briefs, 1):
    print(f"   {i}. {b['id']} - {b['industry']}")
print()

# ═══════════════════════════════════════════════════════════
# 3. دوال مساعدة
# ═══════════════════════════════════════════════════════════

def build_prompts(brief):
    """يبني الـ prompts بشكل موحد لكل النماذج"""
    user_prompt = USER_PROMPT_TEMPLATE.format(
        industry=brief['industry'],
        target_audience=brief['target_audience'],
        brand_purpose=brief['brand_purpose'],
        personality=', '.join(brief['personality']),
        tone=brief['tone']
    )
    return SYSTEM_PROMPT, user_prompt

def clean_json_output(text):
    """ينظف الإخراج ويحاول استخراج JSON صحيح"""
    if not text:
        return None
    
    # إزالة علامات التفكير
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # إزالة markdown
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = text.strip()
    
    # محاولة استخراج JSON من النص
    try:
        # إذا النص كامل JSON
        return json.loads(text)
    except:
        # محاولة استخراج JSON من داخل النص
        json_match = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except:
                return None
        return None

def evaluate_result(json_obj):
    """يقيم جودة الـ JSON"""
    if not json_obj:
        return 0, "فشل - لا JSON"
    
    required_fields = ["brand_name", "tagline", "color_palette", 
                      "typography", "visual_style", "personality_traits"]
    
    score = 0
    for field in required_fields:
        if field in json_obj and json_obj[field]:
            score += 1
    
    return score, f"{score}/{len(required_fields)} حقول"

# ═══════════════════════════════════════════════════════════
# 4. اختبار Mercury 2.5 (عبر API)
# ═══════════════════════════════════════════════════════════

print("="*60)
print("🤖 اختبار Mercury 2.5 (Inception Labs API)")
print("="*60)

user_secrets = UserSecretsClient()
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")
mercury_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key
)

all_results = []
mercury_failed = 0

for i, brief in enumerate(selected_briefs, 1):
    system_p, user_p = build_prompts(brief)
    print(f"\n[{i}/{NUM_BRIEFS}] {brief['id']} - {brief['industry']}")
    
    try:
        response = mercury_client.chat.completions.create(
            model="mercury-2.5",
            messages=[
                {"role": "system", "content": system_p},
                {"role": "user", "content": user_p}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            reasoning_effort="low"
        )
        
        raw = response.choices[0].message.content
        json_obj = clean_json_output(raw)
        score, detail = evaluate_result(json_obj)
        
        if json_obj:
            print(f"   ✅ نجح | التقييم: {detail}")
            print(f"   📝 الاسم: {json_obj.get('brand_name', 'N/A')}")
        else:
            print(f"   ⚠️ فشل استخراج JSON")
            mercury_failed += 1
        
        all_results.append({
            "brief_id": brief["id"],
            "industry": brief["industry"],
            "model": "Mercury-2.5",
            "raw_output": raw,
            "json_output": json_obj,
            "score": score,
            "detail": detail,
            "platform": "API"
        })
        
        time.sleep(0.3)  # rate limiting
        
    except Exception as e:
        print(f"   ❌ خطأ: {str(e)[:80]}")
        mercury_failed += 1
        all_results.append({
            "brief_id": brief["id"],
            "industry": brief["industry"],
            "model": "Mercury-2.5",
            "raw_output": None,
            "json_output": None,
            "score": 0,
            "detail": f"خطأ: {type(e).__name__}",
            "platform": "API"
        })

print(f"\n📊 Mercury: نجح {NUM_BRIEFS - mercury_failed}/{NUM_BRIEFS}")

# ═══════════════════════════════════════════════════════════
# 5. اختبار النماذج المحلية (Qwen + Liquid)
# ═══════════════════════════════════════════════════════════

local_models = [
    {"name": "Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

for model_info in local_models:
    print("\n" + "="*60)
    print(f"🤖 اختبار {model_info['name']} (محلي على GPU)")
    print("="*60)
    
    print(f"⏳ تحميل {model_info['name']}...")
    model = None
    tokenizer = None
    model_failed = 0
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )
        print(f"✅ تم التحميل بنجاح\n")
        
        for i, brief in enumerate(selected_briefs, 1):
            system_p, user_p = build_prompts(brief)
            print(f"[{i}/{NUM_BRIEFS}] {brief['id']} - {brief['industry']}")
            
            try:
                messages = [
                    {"role": "system", "content": system_p},
                    {"role": "user", "content": user_p}
                ]
                
                # معالجة خاصة لـ Qwen (تعطيل التفكير)
                try:
                    inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt",
                        return_dict=True,
                        enable_thinking=False
                    ).to(model.device)
                except TypeError:
                    inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt",
                        return_dict=True
                    ).to(model.device)
                
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    do_sample=True
                )
                
                input_len = inputs["input_ids"].shape[-1]
                raw = tokenizer.decode(
                    outputs[0][input_len:],
                    skip_special_tokens=True
                )
                
                json_obj = clean_json_output(raw)
                score, detail = evaluate_result(json_obj)
                
                if json_obj:
                    print(f"   ✅ نجح | التقييم: {detail}")
                    print(f"   📝 الاسم: {json_obj.get('brand_name', 'N/A')}")
                else:
                    print(f"   ⚠️ فشل استخراج JSON")
                    model_failed += 1
                
                all_results.append({
                    "brief_id": brief["id"],
                    "industry": brief["industry"],
                    "model": model_info["name"],
                    "raw_output": raw,
                    "json_output": json_obj,
                    "score": score,
                    "detail": detail,
                    "platform": "محلي"
                })
                
            except Exception as e:
                print(f"   ❌ خطأ: {str(e)[:80]}")
                model_failed += 1
                all_results.append({
                    "brief_id": brief["id"],
                    "industry": brief["industry"],
                    "model": model_info["name"],
                    "raw_output": None,
                    "json_output": None,
                    "score": 0,
                    "detail": f"خطأ: {type(e).__name__}",
                    "platform": "محلي"
                })
        
        print(f"\n📊 {model_info['name']}: نجح {NUM_BRIEFS - model_failed}/{NUM_BRIEFS}")
        
    except Exception as e:
        print(f"❌ فشل تحميل النموذج: {type(e).__name__}: {e}")
    
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print("🧹 تم تنظيف الذاكرة")

# ═══════════════════════════════════════════════════════════
# 6. حفظ النتائج والتحليل النهائي
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("💾 حفظ النتائج والتحليل النهائي")
print("="*60)

df = pd.DataFrame(all_results)
df.to_csv("fair_comparison_results.csv", index=False)

# حفظ النتائج مع JSON كامل
with open("fair_comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("✅ تم حفظ النتائج في:")
print("   📄 fair_comparison_results.csv")
print("   📄 fair_comparison_results.json\n")

# ملخص المقارنة
print("="*60)
print("📊 ملخص المقارنة العادلة")
print("="*60)

summary = df.groupby('model').agg({
    'score': ['mean', 'sum'],
    'brief_id': 'count'
}).round(2)

print("\n📈 النتائج:")
for model in df['model'].unique():
    model_data = df[df['model'] == model]
    avg_score = model_data['score'].mean()
    success = (model_data['score'] > 0).sum()
    perfect = (model_data['score'] == 6).sum()
    
    print(f"\n🤖 {model}:")
    print(f"   ✅ نجح: {success}/{NUM_BRIEFS}")
    print(f"   🎯 نتائج كاملة: {perfect}/{NUM_BRIEFS}")
    print(f"   📊 متوسط التقييم: {avg_score:.1f}/6")

print("\n🏆 الترتيب حسب الأداء:")
ranking = df.groupby('model')['score'].mean().sort_values(ascending=False)
for i, (model, score) in enumerate(ranking.items(), 1):
    print(f"   {i}. {model}: {score:.2f}/6")

# عرض عينة من النتائج
print("\n" + "="*60)
print("🔍 عينة من النتائج")
print("="*60)

for brief_id in selected_briefs[0:1]:  # أول بريف فقط
    print(f"\n📋 Brief: {brief_id['id']} - {brief_id['industry']}")
    brief_results = df[df['brief_id'] == brief_id['id']]
    for _, row in brief_results.iterrows():
        print(f"\n  🤖 {row['model']}:")
        if row['json_output']:
            print(f"     الاسم: {row['json_output'].get('brand_name', 'N/A')}")
            print(f"     Tagline: {row['json_output'].get('tagline', 'N/A')}")
            print(f"     التقييم: {row['detail']}")
        else:
            print(f"     ❌ {row['detail']}")

print("\n✨ انتهت المقارنة العادلة!")

🚀 بدء المقارنة العادلة بين النماذج الثلاثة

📥 تحميل البيانات من GitHub...
✅ تم اختيار 5 بريفات للمقارنة:
   1. BR001 - coffee
   2. BR002 - healthy snacks
   3. BR003 - specialty bakery
   4. BR004 - specialty tea
   5. BR005 - street food

🤖 اختبار Mercury 2.5 (Inception Labs API)

[1/5] BR001 - coffee
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Nexus Brew

[2/5] BR002 - healthy snacks
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: FuelFit

[3/5] BR003 - specialty bakery
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Crumb & Co.

[4/5] BR004 - specialty tea
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Meridian Leaf

[5/5] BR005 - street food
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: SnapBite

📊 Mercury: نجح 5/5

🤖 اختبار Qwen3-8B (محلي على GPU)
⏳ تحميل Qwen3-8B...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✅ تم التحميل بنجاح

[1/5] BR001 - coffee
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Bean & Brew
[2/5] BR002 - healthy snacks
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: VitalBite
[3/5] BR003 - specialty bakery
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Bakery & Beyond
[4/5] BR004 - specialty tea
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: Cultured Infusions
[5/5] BR005 - street food
   ✅ نجح | التقييم: 6/6 حقول
   📝 الاسم: FlavorFrenzy

📊 Qwen3-8B: نجح 5/5
🧹 تم تنظيف الذاكرة

🤖 اختبار LFM2.5-2.6B (محلي على GPU)
⏳ تحميل LFM2.5-2.6B...


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

✅ تم التحميل بنجاح

[1/5] BR001 - coffee
   ⚠️ فشل استخراج JSON
[2/5] BR002 - healthy snacks
   ⚠️ فشل استخراج JSON
[3/5] BR003 - specialty bakery
   ⚠️ فشل استخراج JSON
[4/5] BR004 - specialty tea
   ⚠️ فشل استخراج JSON
[5/5] BR005 - street food
   ⚠️ فشل استخراج JSON

📊 LFM2.5-2.6B: نجح 0/5
🧹 تم تنظيف الذاكرة

💾 حفظ النتائج والتحليل النهائي
✅ تم حفظ النتائج في:
   📄 fair_comparison_results.csv
   📄 fair_comparison_results.json

📊 ملخص المقارنة العادلة

📈 النتائج:

🤖 Mercury-2.5:
   ✅ نجح: 5/5
   🎯 نتائج كاملة: 5/5
   📊 متوسط التقييم: 6.0/6

🤖 Qwen3-8B:
   ✅ نجح: 5/5
   🎯 نتائج كاملة: 5/5
   📊 متوسط التقييم: 6.0/6

🤖 LFM2.5-2.6B:
   ✅ نجح: 0/5
   🎯 نتائج كاملة: 0/5
   📊 متوسط التقييم: 0.0/6

🏆 الترتيب حسب الأداء:
   1. Mercury-2.5: 6.00/6
   2. Qwen3-8B: 6.00/6
   3. LFM2.5-2.6B: 0.00/6

🔍 عينة من النتائج

📋 Brief: BR001 - coffee

  🤖 Mercury-2.5:
     الاسم: Nexus Brew
     Tagline: Fuel Your Focus
     التقييم: 6/6 حقول

  🤖 Qwen3-8B:
     الاسم: Bean & Brew
     Tagline: Fuel Your 